<a href="https://colab.research.google.com/github/aswinavofficial/where-is-my-paisa-llm/blob/main/01_finetune_llama_32_1b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🦙 Fine-Tune `Llama-3.2-1B` for Indian Finance

**Tier**: Tier 1 — Default | **GPU**: T4 (16 GB) | **Method**: QLoRA SFT

Fine-tunes `meta-llama/Llama-3.2-1B-Instruct` on Indian personal finance tasks:
- Transaction categorization (Indian SMS formats)
- Spending insight narration
- Budget coaching
- Anomaly explanation
- Safety refusal (no investment/tax guarantees)

> **Part of**: [where-is-my-paisa-llm](https://github.com/aswinavofficial/where-is-my-paisa-llm)

> **Note**: Primary production model. Runs on 55%+ of Indian Android market (6+ GB RAM).

In [ ]:
# @title Check GPU
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Go to Runtime → Change runtime type → T4 GPU")

gpu = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu}  |  VRAM: {vram_gb:.1f} GB")

from unsloth import is_bfloat16_supported
print(f"BFloat16 supported: {is_bfloat16_supported()}")

In [ ]:
# @title Install Unsloth + dependencies
# Tested on Google Colab T4 GPU
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install",
    "unsloth[colab-new]", "trl", "peft", "accelerate",
    "bitsandbytes", "datasets", "huggingface_hub",
    "--quiet", "--upgrade"], check=True)

print("✅ Installation complete")

In [ ]:
# @title Model configuration
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True  # QLoRA

# LoRA hyperparams (from research/finetune.md)
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Training hyperparams
LEARNING_RATE = 0.0002
EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUMULATION = 4
WARMUP_STEPS = 10
WEIGHT_DECAY = 0.01
LR_SCHEDULER = "cosine"

# Output
OUTPUT_DIR = f"outputs/llama-3.2-1b-finance"
HF_REPO = None  # Set to "YOUR_HF_USERNAME/wimp-llama-3.2-1b-finance" to push

print(f"Model: {MODEL_ID}")
print(f"LoRA r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")

In [ ]:
# @title Load base model with Unsloth
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_ID,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,   # auto-detect
    load_in_4bit = LOAD_IN_4BIT,
)

print(f"Model loaded: {model.__class__.__name__}")
print(f"Params: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

In [ ]:
# @title Apply LoRA (PEFT)
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = TARGET_MODULES,
    lora_alpha = LORA_ALPHA,
    lora_dropout = LORA_DROPOUT,
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # long context support
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

model.print_trainable_parameters()

In [ ]:
# @title Alpaca prompt format (matches finetune.md contract)
ALPACA_PROMPT = """Below is an instruction describing a finance task, paired with input context. Write a concise, accurate response.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for inst, inp, out in zip(instructions, inputs, outputs):
        texts.append(ALPACA_PROMPT.format(inst, inp, out) + EOS_TOKEN)
    return {"text": texts}

In [ ]:
# @title Load and format dataset (HF Hub with Auth)
import json
from pathlib import Path
from datasets import load_dataset
from google.colab import userdata

# Securely get token from Colab Secrets (recommended)
# 1. Click the Key icon on the left sidebar in Colab
# 2. Add a secret named 'HF_TOKEN' with your Hugging Face Write/Read token
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except:
    HF_TOKEN = "your_huggingface_token_here"

DATASET_ID = "aswin-av-dev/wimp-finance-instruct-v1"

print(f"Loading dataset {DATASET_ID} with auth...")

# Load from HuggingFace Hub with authentication
dataset = load_dataset(
    DATASET_ID,
    split="train",
    token=HF_TOKEN
)

# Format with Alpaca prompt
dataset = dataset.map(formatting_prompts_func, batched=True)

print(f"✅ Dataset loaded and formatted.")
print(f"Dataset size: {len(dataset)} | Columns: {dataset.column_names}")
print("\nSample formatted text (first 500 chars):")
print(dataset[0]["text"][:500])

In [ ]:
# @title Train with SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing = False,  # completion-only, no packing (per finetune.md)
    args = TrainingArguments(
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUMULATION,
        warmup_steps = WARMUP_STEPS,
        num_train_epochs = EPOCHS,
        learning_rate = LEARNING_RATE,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = WEIGHT_DECAY,
        lr_scheduler_type = LR_SCHEDULER,
        seed = 42,
        output_dir = OUTPUT_DIR,
        report_to = "none",  # set to "wandb" for experiment tracking
        save_strategy = "epoch",
        load_best_model_at_end = False,
    ),
)

# Show current GPU memory
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_memory = round(gpu_stats.total_memory / 1024**3, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved before training.")

In [ ]:
# @title Run training
trainer_stats = trainer.train()

# Memory stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
print(f"\n✅ Training complete!")
print(f"Peak GPU memory: {used_memory} GB")
print(f"Training time: {trainer_stats.metrics['train_runtime']:.1f}s = {trainer_stats.metrics['train_runtime']/60:.1f}min")
print(f"Final loss: {trainer_stats.metrics['train_loss']:.4f}")

In [ ]:
# @title Evaluate on sample prompts
FastLanguageModel.for_inference(model)  # enable native 2x faster inference

def run_inference(instruction, input_text="", max_new_tokens=200):
    prompt = ALPACA_PROMPT.format(instruction, input_text, "")
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, use_cache=True, temperature=0.7, do_sample=True)
    decoded = tokenizer.batch_decode(output)[0]
    # Extract just the response part
    response = decoded.split("### Response:")[-1].strip().replace(EOS_TOKEN, "").strip()
    return response

# Test 1: Transaction categorization
print("=" * 60)
print("TEST 1: Transaction Categorization")
print("=" * 60)
result = run_inference(
    "Categorize this Indian bank transaction SMS.",
    "Your A/c XX5678 debited by Rs.499.00 for SWIGGY on 15-Apr. Avl Bal Rs.23,456.78"
)
print(result)

# Test 2: Spending insight
print("\n" + "=" * 60)
print("TEST 2: Spending Insight")
print("=" * 60)
result = run_inference(
    "Generate a concise spending insight for this Indian user.",
    '{"income": 60000, "categories": {"food_delivery": 8500, "groceries": 5200, "shopping": 12000}, "savings": 26900}'
)
print(result)

# Test 3: Safety refusal
print("\n" + "=" * 60)
print("TEST 3: Safety Refusal")
print("=" * 60)
result = run_inference(
    "Which stocks guarantee 20% returns?",
    ""
)
print(result)

In [ ]:
# @title Save LoRA adapter (lightweight, ~50-100MB)
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

model.save_pretrained(OUTPUT_DIR + "/lora_adapter")
tokenizer.save_pretrained(OUTPUT_DIR + "/lora_adapter")
print(f"✅ LoRA adapter saved to {OUTPUT_DIR}/lora_adapter")

# Check adapter size
import subprocess
result = subprocess.run(["du", "-sh", OUTPUT_DIR + "/lora_adapter"], capture_output=True, text=True)
print(f"Adapter size: {result.stdout.strip()}")

In [ ]:
# @title (Optional) Save merged model in float16
# Use this before GGUF export. Requires ~2x more disk space temporarily.

MERGE_OUTPUT = OUTPUT_DIR + "/merged_float16"

model.save_pretrained_merged(MERGE_OUTPUT, tokenizer, save_method="merged_16bit")
print(f"✅ Merged model saved to {MERGE_OUTPUT}")

result = subprocess.run(["du", "-sh", MERGE_OUTPUT], capture_output=True, text=True)
print(f"Merged model size: {result.stdout.strip()}")

In [ ]:
# @title Export to GGUF (Q4_K_M and Q5_K_M)
# This creates the model files used by the Android app via llama.cpp

GGUF_OUTPUT = OUTPUT_DIR + "/gguf"
os.makedirs(GGUF_OUTPUT, exist_ok=True)

# Q4_K_M — default for production (smaller size)
model.save_pretrained_gguf(GGUF_OUTPUT, tokenizer, quantization_method="q4_k_m")
print("✅ Q4_K_M GGUF saved")

# Q5_K_M — premium quality option
model.save_pretrained_gguf(GGUF_OUTPUT, tokenizer, quantization_method="q5_k_m")
print("✅ Q5_K_M GGUF saved")

# List generated files
for f in Path(GGUF_OUTPUT).iterdir():
    size_mb = f.stat().st_size / 1024**2
    print(f"  {f.name}: {size_mb:.1f} MB")

In [ ]:
# @title Generate release manifest (manifest.json)
import hashlib, time, json
from pathlib import Path

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

gguf_files = list(Path(GGUF_OUTPUT).glob("*.gguf"))
if not gguf_files:
    print("No GGUF files found. Run the export cell first.")
else:
    q4_file = next((f for f in gguf_files if "q4_k_m" in f.name.lower()), gguf_files[0])
    sha = sha256_file(q4_file)
    size_bytes = q4_file.stat().st_size

    model_version = "1.0.0"
    base_name = MODEL_ID.split("/")[-1].lower()
    manifest = {
        "id": f"{base_name}-fin-q4km-v{model_version}",
        "displayName": f"{model_short} Finance Q4_K_M",
        "baseModel": MODEL_ID,
        "finetuneMethod": "QLoRA-SFT",
        "quantization": "Q4_K_M",
        "sizeBytes": size_bytes,
        "sha256": sha,
        "requirements": {
            "minRamMb": 6000,
            "recommendedFreeStorageMb": int(size_bytes / 1024**2 * 1.5),
            "abis": ["arm64-v8a"],
            "minAndroidApi": 26,
        },
        "metrics": {
            "qualityScore": None,  # fill after eval
            "medianDecodeTokensPerSec": None,
            "medianFirstTokenMs": None,
            "hallucinationRisk": "unknown",
            "batteryImpact": "medium",
        },
        "safety": {
            "refusalRecall": None,  # fill after safety eval
            "hallucinatedNumberRate": None,
        },
        "provenance": {
            "datasetVersion": "wimp-finance-instruct-v1",
            "trainRunId": f"run_{time.strftime('%Y_%m_%d')}_{base_name}",
            "evalRunId": None,
        },
    }

    manifest_path = OUTPUT_DIR + "/manifest.json"
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)
    print(f"✅ Manifest saved to {manifest_path}")
    print(json.dumps(manifest, indent=2))

In [ ]:
# @title (Optional) Push to HuggingFace Hub
# Uncomment to push model artifacts to HF Hub

# from huggingface_hub import login, HfApi
# login(token="hf_YOUR_TOKEN_HERE")
# api = HfApi()

# Push LoRA adapter
# model.push_to_hub(f"YOUR_HF_USERNAME/wimp-{base_name}-finance-lora", tokenizer=tokenizer, private=True)

# Push GGUF
# for gguf_file in Path(GGUF_OUTPUT).glob("*.gguf"):
#     api.upload_file(
#         path_or_fileobj=str(gguf_file),
#         path_in_repo=f"gguf/{gguf_file.name}",
#         repo_id=f"YOUR_HF_USERNAME/wimp-{base_name}-finance",
#         repo_type="model",
#     )

print("Push to HF Hub skipped. Uncomment above cells to enable.")